In [ ]:
# Transformers undet the hood: Attention is all you need (https://arxiv.org/pdf/1706.03762)

# GPT: Generativaly Pre-trained Transformers

In [ ]:
# Dataset: Tiny Shakesphere

In [ ]:
!pwd
!ls

/content
drive  sample_data


In [ ]:
%cd drive/MyDrive/Colab\ Notebooks/Transformers
!ls

/content/drive/MyDrive/Colab Notebooks/Transformers
GPT.ipynb  input.txt


In [ ]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
print("char length: ",len(text))

char length:  1115393


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [ ]:
chars = sorted(list(set(text))) # sorted list of all the char that occures
vocab_size = len(chars)  # size of vocabulary
print(''.join(chars))
print(vocab_size)

# 0 is '\n' and 1 is ''


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
# tokenization of input text (vocab to numbers)
# char level lang model
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of int, output a string

print(encode("hi there"))
print(decode(encode("hi there")))

# others:  openAI-tiktokens, google-Sentencebit

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [ ]:
# Encoding entire dataset

import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115393]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [ ]:
# data split (tr: 90%, test:10%)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
# how we train a transformer:
    # To compute heavy to do it all at once, so we sample random chunks and train iterativily on them
    # max len of chunks ( block_size )

In [ ]:
# how transformers work? unlike RNNs and CNNs, transformers train the block_size in parallel not iterative
# So, a block size of 8 means, training 8 threads starting with a pair (1 prev predicts next) then (2 prev predicts
# next) ... total 8

block_size = 8
train_data[:block_size+1] # why +1, see above

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [ ]:
# This is done so that transformers learns connections from as little as 1 input to block_size inputs
# So that sampling can be done from the begining without any reference

# this is the Time dimension: what comes next (pridicting future char)
x = train_data[:block_size] #(0, 7)
y = train_data[1:block_size + 1] #(1, 8)
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target is: {target}")

when input is tensor([18]) the target is: 47
when input is tensor([18, 47]) the target is: 56
when input is tensor([18, 47, 56]) the target is: 57
when input is tensor([18, 47, 56, 57]) the target is: 58
when input is tensor([18, 47, 56, 57, 58]) the target is: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is: 58


In [ ]:
# next dimension Batch dimension:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
  # genereate a small batch
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))  # where the batch starts
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

for b in range(batch_size): # batch dim
  for t in range(block_size): # time dim
    context = xb[b, :t+1] # input to tranformer
    target = yb[b, t] # what to expect out of transformer
    print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[53, 59,  6,  1, 58, 56, 47, 40],
        [49, 43, 43, 54,  1, 47, 58,  1],
        [13, 52, 45, 43, 50, 53,  8,  0],
        [ 1, 39,  1, 46, 53, 59, 57, 43]])
targets:
torch.Size([4, 8])
tensor([[59,  6,  1, 58, 56, 47, 40, 59],
        [43, 43, 54,  1, 47, 58,  1, 58],
        [52, 45, 43, 50, 53,  8,  0, 26],
        [39,  1, 46, 53, 59, 57, 43,  0]])
when input is [53] the target: 59
when input is [53, 59] the target: 6
when input is [53, 59, 6] the target: 1
when input is [53, 59, 6, 1] the target: 58
when input is [53, 59, 6, 1, 58] the target: 56
when input is [53, 59, 6, 1, 58, 56] the target: 47
when input is [53, 59, 6, 1, 58, 56, 47] the target: 40
when input is [53, 59, 6, 1, 58, 56, 47, 40] the target: 59
when input is [49] the target: 43
when input is [49, 43] the target: 43
when input is [49, 43, 43] the target: 54
when input is [49, 43, 43, 54] the target: 1
when input is [49, 43, 43, 54, 1] the target: 47
when input is [49, 43, 43, 5

In [36]:
# implementing a bigram language model
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigamLanaguageModel(nn.Module):  # BigamLanaguageModel is a subclass of nn.Module

  def __init__(self, vocab_size):
    super().__init__()
    # each token directly reads off the logits for the next token from the table lookup
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # creates a wrapper aroung tensor of given size

  def forward(self, idx, targets=None):  # method over-ridding ( defining how forward pass will happen)

    # idx ad targets are both (B, T) tensor of intergets
    logits = self.token_embedding_table(idx) # (B, T, C) (batch = batch_size, time = block_size, channels = vocab_size)

    if targets == None:
      loss = None
    else:

      # how to pass logits to F.cross_entropy? we have (B, T, C), F.cross_entropy expects C to be 2nd dim
      B, T, C = logits.shape
      logits = logits.view(B*T, C) # (4, 8, 65) -> (32, 65)
      targets = targets.view(B*T) # (4, 8) -> (32)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):
      # get the predictions
      logits, loss = self(idx) # perfroms __call__ on model
      # focus only on last time step
      logits = logits[ :, -1 ,: ] # becomes (B, C) : to keep it bigram (only look at just previous not the whole history)
      # apply softmax to get probs
      probs = F.softmax(logits, dim=1) # (B, C)
      # sample from the distribution
      idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
      # append sampled idx to the running sequence
      idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
    return idx


m = BigamLanaguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8948, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [37]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)  # what we used till now = SGT Schotastic gradient descent, now AdamW

In [43]:
batch_size = 32
for steps in range(10000):

  # sample a batch of date
  xb, yb = get_batch('train') # (B, T, C) = (32, 8, 65)

  # evaluate the loss
  logits, loss = m(xb, yb) # forward pass
  optimizer.zero_grad(set_to_none=True) # gards to None
  loss.backward() # backward pass
  optimizer.step() # update the grads

print(loss.item())

2.535524606704712


In [45]:
print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


L&ZATher isalon stary RKEOnores us w be g? bagof tts! at ame owo'd bandre,
JUMonono trawnthetrid ang pat
My t nco grwinste t w rtce,

NIDULIRGorr best gerer, nt; co hamyose EESof bun 'APAExcowhe;
UCHAnd d bumod
Ghacendvevesou mbouk'su mbam

S:
OMIUCoake:
them man.
BEYOMETht l at
Sun st tite wit we t mpt mamas lissherrd tond murd sug, wrode.
Thackinonour'se Per s wanothepaf-
AUFowaid anixat:
KEDWas aucund me bye d besw Gound s s:
Bueromng stou d s or brvead avant Afougshtinofar t situtixe arausy 
